#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("speech.txt")
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=1000,chunk_overlap=30)
docs = text_splitter.split_documents(documents)

In [4]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct our

In [6]:
embedding = OllamaEmbeddings(model="nomic-embed-text")
db=FAISS.from_documents(docs,embedding)
db

In [9]:
### Querying 

query = "What does the speaker believe is the main reason the United States should enter the war?"
docs = db.similarity_search(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [11]:
retriver=db.as_retriever()
docs=retriver.invoke(query)

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance(Manhatten Distance). Therefore, a lower score is better.

In [13]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='611d9434-be14-4fd1-accd-bdf26f8b952d', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  np.float32(316.2583)),
 (Document(id='4e375274-9314-47b8-bf68-143f9b8e1e13', metadata={'source': 'spee

In [14]:
embedding_vector=embedding.embed_query(query)

embedding_vector


[0.19701501727104187,
 2.383436918258667,
 -3.2218379974365234,
 -0.49923136830329895,
 1.9613524675369263,
 2.105024576187134,
 -0.6781808137893677,
 -0.31930530071258545,
 1.2243549823760986,
 0.1851721704006195,
 -0.37718552350997925,
 0.4746550917625427,
 0.2091546356678009,
 1.114794135093689,
 1.2488653659820557,
 -0.5341628789901733,
 -0.13787169754505157,
 -1.5660109519958496,
 -0.7663438320159912,
 0.5667570233345032,
 -1.2439706325531006,
 -0.9484353065490723,
 0.22032983601093292,
 -0.7271776795387268,
 1.4182571172714233,
 0.8590003848075867,
 0.0900297462940216,
 0.9354607462882996,
 -0.4429011344909668,
 0.0740736648440361,
 1.3095260858535767,
 -1.2863770723342896,
 -0.06304620951414108,
 0.28018152713775635,
 -1.2858818769454956,
 -1.3696255683898926,
 -0.18344691395759583,
 1.0277451276779175,
 0.0730123370885849,
 -1.091118574142456,
 -0.3854760229587555,
 0.29622772336006165,
 0.31186118721961975,
 -1.5125718116760254,
 1.0559885501861572,
 -0.7430944442749023,
 -0.2

In [15]:
docs_and_score_with_vector = db.similarity_search_with_score_by_vector(embedding_vector)
docs_and_score_with_vector

[(Document(id='611d9434-be14-4fd1-accd-bdf26f8b952d', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  np.float32(316.2583)),
 (Document(id='4e375274-9314-47b8-bf68-143f9b8e1e13', metadata={'source': 'spee

In [17]:
### Saving and loading the VectorStoreDB
db.save_local("faiss_index")

In [21]:
new_db = FAISS.load_local("faiss_index",embedding,allow_dangerous_deserialization=True)

docs = new_db.similarity_search(query)

In [22]:
docs

[Document(id='611d9434-be14-4fd1-accd-bdf26f8b952d', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='4e375274-9314-47b8-bf68-143f9b8e1e13', metadata={'source': 'speech.txt'}, page_content='The